# Analyse de 30 forages : estimer le fer par krigeage et par Machine Learning

Ce notebook reprend, pas à pas, la démarche appliquée à un gisement de BIF du Sud-Cameroun.

> **Données.** Les données réelles appartiennent à MuCoTech SARL et ne sont pas publiées. Ce notebook tourne sur un **jeu synthétique de même structure** : 30 forages, ~3 700 m, ~2 700 échantillons, 3 domaines, coordonnées et orientation fictives. Les chiffres obtenus ici montrent le fonctionnement de la méthode. Les résultats du gisement réel figurent dans le README.

**Plan**
1. Charger et décrire la base de forages
2. Compositer, écrêter, séparer les forages de test
3. Variographie
4. Estimer : krigeage ordinaire, Random Forest, XGBoost, stacking
5. Valider : précision globale, biais conditionnel, variables explicatives
6. Ce qu'il faut retenir

In [1]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, "../src")
import donnees_synthetiques as ds, analyse as an
from sklearn.inspection import permutation_importance
pd.set_option("display.precision", 2)

## 1. La base de forages

In [2]:
brut = ds.generer()
print(f"{brut.forage.nunique()} forages | {brut.groupby('forage').a.max().sum():.0f} m forés | {len(brut)} échantillons")
brut.head()

30 forages | 3679 m forés | 2725 échantillons


,forage,x,y,z,de,a,domaine,Fe,SiO2,Al2O3,P
0,DH001,1009.16,1993.28,253.08,0.00,1.35,ENCAISSANT,2.12,52.49,12.88,0.08
1,DH001,1009.16,1993.28,251.73,1.35,2.70,ENCAISSANT,4.20,53.20,13.96,0.14
2,DH001,1009.16,1993.28,250.38,2.70,4.05,ENCAISSANT,3.04,57.07,13.05,0.07
3,DH001,1009.16,1993.28,249.03,4.05,5.40,ENCAISSANT,3.99,55.32,11.73,0.14
4,DH001,1009.16,1993.28,247.68,5.40,6.75,ENCAISSANT,3.04,59.47,11.43,0.03


In [3]:
brut.groupby("domaine")[["Fe", "SiO2", "Al2O3", "P"]].agg(["count", "mean", "std", "min", "max"]).round(2)

Fe                             SiO2                      \
             count   mean   std    min    max count   mean   std    min   
domaine                                                                   
ENCAISSANT    1534   2.10  1.15   1.00   8.22  1534  61.25  3.83  49.33   
ITAB_FRAICHE   923  33.24  4.06  22.88  46.80   923  45.76  5.14  30.06   
ITAB_OXYDEE    268  42.30  6.59  29.36  58.81   268  32.43  5.97  17.63   

                    Al2O3                               P                    \
                max count   mean   std   min    max count  mean   std   min   
domaine                                                                       
ENCAISSANT    70.65  1534  12.97  1.55  6.42  17.86  1534  0.07  0.05  0.01   
ITAB_FRAICHE  56.47   923   2.61  1.40  0.20   7.31   923  0.07  0.04  0.01   
ITAB_OXYDEE   42.74   268   2.57  1.39  0.20   7.15   268  0.07  0.05  0.01   

                    
               max  
domaine             
ENCAISSANT    0.67  
ITAB_FRAICHE  0.35  
ITAB_OXYDEE   0.42

![Implantation](../figures/01_implantation_forages.png)
![Exploration](../figures/02_exploration_donnees.png)

**Lecture.** Les trois domaines ont des distributions de fer bien séparées. Mélanger l'encaissant et l'itabirite dans une même estimation produirait une moyenne sans signification géologique, d'où l'**estimation par domaine**. Dans l'itabirite, le fer varie en sens inverse de la silice : c'est la dilution du minerai par le quartz des bandes cherteuses.

## 2. Préparation : compositage, écrêtage, jeu de test

In [4]:
comp = an.composites(brut, longueur=2.0)
comp = comp[comp.domaine.isin(an.DOMAINES_MINERALISES)].reset_index(drop=True)
comp["dom_oxyde"] = (comp.domaine == "ITAB_OXYDEE").astype(int)
comp["P"], seuil = an.top_cut(comp["P"], 0.995)
train, test, test_ids = an.separer_par_forage(comp, part_test=0.2)
print(f"{len(comp)} composites de 2 m | top-cut P à {seuil:.3f} %")
print(f"Entraînement : {train.forage.nunique()} forages | Test : {test.forage.nunique()} forages -> {test_ids}")

822 composites de 2 m | top-cut P à 0.245 %
Entraînement : 24 forages | Test : 6 forages -> ['DH003', 'DH013', 'DH018', 'DH021', 'DH026', 'DH029']


**Pourquoi tenir à l'écart des forages entiers ?** Deux échantillons consécutifs d'un même trou sont presque identiques. Un tirage aléatoire d'échantillons mettrait des voisins immédiats dans l'entraînement et dans le test, et le score serait gonflé. Tenir à l'écart des forages complets reproduit la vraie question : *que vaut l'estimation là où l'on n'a pas foré ?*

## 3. Variographie

In [5]:
fraiche = comp[comp.domaine == "ITAB_FRAICHE"]
an.variogramme_directionnel(fraiche, "Fe", an.AZIMUT).head(8)

,distance,gamma,paires
0,77.88,5.07,1551
1,104.97,5.58,2765
2,129.71,5.54,747
3,176.36,6.16,827
4,197.46,9.56,4969
5,224.59,9.73,2231
6,259.20,8.74,515
7,291.67,11.98,3524


![Variogrammes](../figures/03_variogrammes.png)

Sur ce jeu fictif, la continuité est plus longue selon l'axe majeur (60°) que dans la direction perpendiculaire. Ces portées dimensionnent l'ellipsoïde de recherche du krigeage. Sur le gisement réel, l'analyse menée sous Isatis.neo a donné une anisotropie NW-SE (~135°), un effet de pépite quasi nul et des portées de 150 à 300 m selon le domaine.

## 4. Estimation

In [6]:
preds = {"Krigeage ordinaire": an.krigeage_ordinaire(train, test)[0]}
ml, modeles = an.entrainer_et_predire(train, test)
preds.update(ml)
resultats = pd.DataFrame({n: an.metriques(test.Fe, p) for n, p in preds.items()}).T
resultats.round(3)

,R2,RMSE,MAE,Biais moyen,Pente de régression
Krigeage ordinaire,0.78,2.79,2.20,-0.63,1.04
Random Forest,0.94,1.51,1.21,-0.01,1.04
XGBoost,0.93,1.62,1.29,-0.12,1.07
Stacking RF+XGB,0.94,1.51,1.21,0.02,1.03


![Observé vs estimé](../figures/04_observe_vs_estime.png)

Le nuage du krigeage est **aplati** : les fortes teneurs sont sous-estimées et les faibles surestimées. C'est l'effet de lissage. Les modèles ML suivent mieux la diagonale, parce qu'ils exploitent la chimie de l'échantillon (SiO₂ en tête) en plus de sa position.

## 5. Validation

In [7]:
an.biais_par_classe(test.Fe.to_numpy(), preds).round(2)

,Observé,Krigeage ordinaire,Random Forest,XGBoost,Stacking RF+XGB
classe,,,,,
"(0, 27]",25.95,30.24,28.43,27.98,28.11
"(27, 30]",28.84,31.44,29.83,29.98,29.83
"(30, 33]",31.64,31.65,32.05,31.93,31.98
"(33, 36]",34.46,34.12,34.47,34.51,34.54
"(36, 40]",37.70,36.32,37.35,37.27,37.44
"(40, 100]",46.55,43.13,45.49,44.99,45.59


![Biais conditionnel](../figures/05_biais_conditionnel.png)

In [8]:
perm = permutation_importance(modeles["Stacking RF+XGB"], test[an.VARIABLES], test.Fe,
                              n_repeats=20, random_state=an.SEED, scoring="r2")
pd.Series(perm.importances_mean, index=an.VARIABLES).sort_values(ascending=False).round(3)

SiO2         1.40e+00
Al2O3        3.50e-02
y            1.80e-02
dom_oxyde    6.00e-03
z            6.00e-03
x            5.00e-03
P            3.00e-03
dtype: float64

![Importance](../figures/06_importance_variables.png)

## 6. Ce qu'il faut retenir

1. **Le Machine Learning améliore la précision locale** et réduit le biais conditionnel, surtout dans les classes de teneur extrêmes.
2. **Ce gain vient surtout de SiO₂.** Pour estimer un modèle de blocs, la silice doit elle-même être estimée en chaque bloc (krigeage ou co-krigeage), et son erreur s'ajoute. Le gain réel sur blocs est donc inférieur au gain mesuré sur échantillons.
3. **Le krigeage reste la référence de reporting.** Il est traçable et auditable, et la classification JORC s'appuie sur ses sorties (efficacité de krigeage, variance). La meilleure pratique est hybride : le krigeage pour le cadre, le ML pour affiner et contrôler.

*Résultats du gisement réel (mémoire) : R² = 0,606 en krigeage ordinaire et 0,825 en stacking ; 102,1 Mt à 31,4 % Fe, dont 58,2 Mt en Mesurée + Indiquée (JORC 2012).*